# Random Forest Model - Training & Evaluation
This notebook trains a Random Forest classifier on fuel order data to predict cancellations.

In [4]:
import sys, json, os, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
import joblib

from datetime import datetime

## 1. Load & Explore Data

In [5]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
CSV_PATH = os.path.join(BASE_DIR, 'data', 'fuelsewa_synthetic_orders.csv')

df = pd.read_csv(CSV_PATH)
print(f'Total samples: {len(df)}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nClass distribution:')
print(df['status'].value_counts())

Total samples: 800

Columns: ['order_id', 'userId', 'fuelType', 'quantity', 'deliveryLocation_latitude', 'deliveryLocation_longitude', 'deliveryLocation_address', 'requestSource', 'pricing_pricePerLiter', 'pricing_fuelCost', 'pricing_deliveryFee', 'pricing_emergencyFee', 'pricing_totalPrice', 'status', 'priority', 'isEmergency', 'isFarZone', 'cancelReason', 'estimatedDeliveryMinutes', 'distance_km', 'createdAt']

Class distribution:
status
delivered    546
cancelled    254
Name: count, dtype: int64


In [6]:
df.head()

,order_id,userId,fuelType,quantity,deliveryLocation_latitude,deliveryLocation_longitude,deliveryLocation_address,requestSource,pricing_pricePerLiter,pricing_fuelCost,...,pricing_emergencyFee,pricing_totalPrice,status,priority,isEmergency,isFarZone,cancelReason,estimatedDeliveryMinutes,distance_km,createdAt
0,0a6b173c7cbe43a2916f6f05,0ea0c96bdb5847ec823d6cd7,petrol,9.6,27.695811,85.289070,"Teku, Kathmandu Valley",roadside,178,1708.8,...,10,1798.8,delivered,high,True,False,NaN,24,3.65,2026-05-29T15:43:00
1,adfd887c47bc446eb2d123f2,11f21bbee0cc433eadd8677b,petrol,18.6,27.735611,85.604599,"Sundarijal, Kathmandu Valley",home,178,3310.8,...,0,4000.8,cancelled,normal,False,True,Repeated poor experiences with unknown drivers,116,34.49,2026-02-15T19:38:00
2,38a2796f97f849fe9f112b0b,c9620180f34141e8bf209b76,diesel,4.5,27.676263,85.230261,"Jamal, Kathmandu Valley",home,163,733.5,...,0,813.5,delivered,normal,False,False,NaN,15,2.98,2026-05-15T13:44:00
3,208d7eb1fa264d2e927b39b3,49c5e185c76444b7bc64a478,diesel,2.7,27.687980,85.271680,"Khumaltar, Kathmandu Valley",home,163,440.1,...,0,520.1,delivered,normal,False,False,NaN,27,1.83,2026-03-19T17:06:00
4,a2b7162b8af24b9ba443b941,fbf271d737c74f1e804aeaa9,diesel,7.0,27.672145,85.259841,"Suryabinayak, Kathmandu Valley",office,163,1141.0,...,0,1221.0,delivered,normal,False,False,NaN,10,0.30,2026-03-20T08:05:00


## 2. Preprocessing

In [7]:
df['target'] = (df['status'] == 'cancelled').astype(int)

for col in ['isEmergency', 'isFarZone']:
    df[col] = df[col].astype(float)

cat_map = {
    'fuelType': {'petrol': 0.0, 'diesel': 1.0},
    'requestSource': {'home': 0.0, 'office': 1.0, 'other': 2.0, 'roadside': 3.0},
    'priority': {'normal': 0.0, 'high': 1.0},
}
for col, mapping in cat_map.items():
    df[col] = df[col].fillna('unknown').astype(str).str.strip().map(mapping).fillna(-1.0)

def extract_hour(ts):
    return float(ts.split('T')[1].split(':')[0])

def extract_dow(ts):
    parts = ts.split('T')[0].split('-')
    y, m, d = int(parts[0]), int(parts[1]), int(parts[2])
    return float(datetime(y, m, d).weekday())

df['hour_of_day'] = df['createdAt'].apply(extract_hour)
df['day_of_week'] = df['createdAt'].apply(extract_dow)
df['is_weekend'] = (df['day_of_week'] >= 5).astype(float)

## 3. Feature Selection

In [8]:
feature_cols = [
    'quantity', 'pricing_totalPrice', 'pricing_deliveryFee',
    'pricing_emergencyFee', 'estimatedDeliveryMinutes', 'distance_km',
    'hour_of_day', 'day_of_week', 'is_weekend',
    'isEmergency', 'isFarZone',
    'fuelType', 'requestSource', 'priority',
]

X = df[feature_cols].values.astype(np.float64)
y = df['target'].values.astype(np.int32)

print(f'Feature matrix shape: {X.shape}')
print(f'Target distribution:')
print(pd.Series(y).value_counts().to_string())

Feature matrix shape: (800, 14)
Target distribution:
0    546
1    254


## 4. Train-Test Split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples: {len(X_train)}')
print(f'Testing samples: {len(X_test)}')
print(f'\nTraining set distribution:')
print(pd.Series(y_train).value_counts().to_string())
print(f'\nTest set distribution:')
print(pd.Series(y_test).value_counts().to_string())

Training samples: 640
Testing samples: 160

Training set distribution:
0    437
1    203

Test set distribution:
0    109
1     51


## 5. Feature Scaling

In [10]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print('Features scaled using StandardScaler')

Features scaled using StandardScaler


## 6. Train Random Forest Model

In [11]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=1,
)
model.fit(X_train_s, y_train)
print('Model trained successfully!')

Model trained successfully!


## 7. Evaluation

In [12]:
y_pred = model.predict(X_test_s)
accuracy = (y_test == y_pred).mean()
f1 = f1_score(y_test, y_pred)

print(f'Accuracy : {accuracy:.3f}')
print(f'F1 Score : {f1:.3f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['delivered', 'cancelled']))

Accuracy : 0.812
F1 Score : 0.706

Classification Report:
              precision    recall  f1-score   support

   delivered       0.86      0.86      0.86       109
   cancelled       0.71      0.71      0.71        51

    accuracy                           0.81       160
   macro avg       0.78      0.78      0.78       160
weighted avg       0.81      0.81      0.81       160



## 8. Confusion Matrix

In [13]:
cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(f'            Predicted')
print(f'            Deliv Canc')
print(f'Actual Deliv {cm[0][0]:5d} {cm[0][1]:5d}')
print(f'       Canc {cm[1][0]:5d} {cm[1][1]:5d}')

Confusion Matrix:
            Predicted
            Deliv Canc
Actual Deliv    94    15
       Canc    15    36


## 9. Feature Importance

In [14]:
importances = model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
print('Feature Importance:')
for i in sorted_idx:
    print(f'  {feature_cols[i]:35s} {importances[i]:.4f}')

Feature Importance:
  pricing_totalPrice                  0.2191
  quantity                            0.1969
  distance_km                         0.1458
  estimatedDeliveryMinutes            0.1125
  pricing_deliveryFee                 0.0993
  isFarZone                           0.0741
  hour_of_day                         0.0574
  day_of_week                         0.0402
  requestSource                       0.0217
  fuelType                            0.0119
  is_weekend                          0.0091
  isEmergency                         0.0047
  priority                            0.0041
  pricing_emergencyFee                0.0033


## 10. Test Prediction Samples

In [15]:
sample_indices = np.random.choice(len(X_test), size=min(10, len(X_test)), replace=False)
print('Sample Predictions (first 10 test samples):')
print(f'{"#":>3s} {"Actual":>8s} {"Predicted":>10s} {"Confidence":>10s}')
print('-' * 35)
for i, idx in enumerate(sample_indices):
    actual = 'cancelled' if y_test[idx] == 1 else 'delivered'
    pred = 'cancelled' if y_pred[idx] == 1 else 'delivered'
    prob = model.predict_proba(X_test_s[idx:idx+1])[0][1]
    print(f'{i:3d} {actual:>8s} {pred:>10s} {prob:>9.1%}')

Sample Predictions (first 10 test samples):
  #   Actual  Predicted Confidence
-----------------------------------
  0 delivered  delivered      1.5%
  1 delivered  delivered      3.5%
  2 cancelled  delivered      2.1%
  3 cancelled  cancelled     69.9%
  4 delivered  delivered      6.0%
  5 delivered  cancelled     50.1%
  6 cancelled  cancelled     98.5%
  7 delivered  delivered     24.0%
  8 cancelled  delivered     49.3%
  9 delivered  delivered     12.0%
